# Heston Calibration, End to End

**A from-scratch walkthrough of the calibration method in this repo** — background, the pricing engine, the analytic Jacobian, Levenberg–Marquardt / Trust-Region-Reflective, how to run it, and where it breaks.

Every block is tied to the actual source files:

| concept | file |
|---|---|
| Characteristic function (Cui 2016) | `models/heston_cf_cui.py` |
| GL-quadrature pricer + analytic gradient | `pricing/european_gl.py` |
| Residuals, Jacobian, vega weights | `calibration/heston_loss_function.py` |
| LM / TRF driver | `calibration/calibrate_heston.py` |
| Universe selection + service entry point | `services/calibration_service.py` |

> Run the code cells top-to-bottom from the repo root (or with the repo on `sys.path`). The math cells render via MathJax in Jupyter / VS Code.

---
## 1. Background: what "calibration" means

Heston has **five unknown parameters**:

$$\theta = (v_0,\ \kappa,\ \bar v,\ \sigma,\ \rho)$$

| symbol | code name | meaning |
|---|---|---|
| $v_0$ | `v0` | variance *now* (today's instantaneous variance) |
| $\kappa$ | `kappa` | mean-reversion speed of variance |
| $\bar v$ | `theta` | long-run variance level |
| $\sigma$ | `sigma` | vol-of-vol (how noisy variance itself is) |
| $\rho$ | `rho` | spot/variance correlation (the skew driver) |

Risk-neutral dynamics:

$$
dS_t = (r-q)\,S_t\,dt + \sqrt{v_t}\,S_t\,dW_t^S,\qquad
dv_t = \kappa(\bar v - v_t)\,dt + \sigma\sqrt{v_t}\,dW_t^v,\qquad
d\langle W^S,W^v\rangle = \rho\,dt.
$$

The market gives you option prices $C^{*}_i$ at $(K_i,T_i)$. **Calibration is the inverse problem**: find $\theta$ that reproduces them.

$$
\boxed{\ \theta^{*} = \arg\min_{\theta}\ \tfrac12\sum_i w_i^2\big(C_{\text{model}}(\theta;K_i,T_i) - C^{*}_i\big)^2\ }
$$

This is a **nonlinear least-squares** problem with three sub-problems:

1. **Forward problem** — given $\theta$, compute $C_{\text{model}}$ fast (done thousands of times).
2. **Sensitivity** — given $\theta$, compute $\partial C_{\text{model}}/\partial\theta_j$ (the Jacobian).
3. **Optimizer** — Levenberg–Marquardt / Trust-Region consuming (1)+(2).

The rest of the notebook is these three, in order.

---
## 2. The forward problem: pricing under Heston

Calibration is "invert the pricer," so the pricer is the heart of it. Heston has no closed-form price, but it has a closed-form **characteristic function** (CF), and a standard Fourier bridge CF → price.

### 2.1 The Fourier pricing identity

Let $x_T=\ln S_T$ and $\varphi(u)=\mathbb E^{Q}[e^{iu x_T}]$ (closed form, next cell).

A European call $C=e^{-rT}\mathbb E[(S_T-K)^+]$ splits into two probabilities:

$$C = S_0 e^{-qT}\,\Pi_1 - K e^{-rT}\,\Pi_2,$$

with $\Pi_2 = Q(S_T>K)$ and $\Pi_1$ the same under the *share* measure. Gil–Pelaez inversion gives each:

$$
\Pi_2 = \tfrac12 + \tfrac1\pi\int_0^\infty \mathrm{Re}\!\left[\frac{K^{-iu}\,\varphi(u)}{iu}\right]du,
\qquad
\Pi_1 = \tfrac12 + \tfrac1\pi\int_0^\infty \mathrm{Re}\!\left[\frac{K^{-iu}\,\varphi(u-i)}{iu\,\varphi(-i)}\right]du.
$$

Using $\varphi(-i)=\mathbb E[S_T/S_0]=e^{(r-q)T}$, the prefactor $S_0 e^{-qT}/\varphi(-i)$ collapses to $e^{-rT}$, landing exactly on the form the code uses:

$$
\boxed{\ C = \tfrac12\big(S_0 e^{-qT} - K e^{-rT}\big) + \frac{e^{-rT}}{\pi}\big(I_1 - K\,I_2\big)\ }
$$

$$
I_1=\int_0^\infty \mathrm{Re}\!\left[\frac{K^{-iu}}{iu}\varphi(u-i)\right]du,
\qquad
I_2=\int_0^\infty \mathrm{Re}\!\left[\frac{K^{-iu}}{iu}\varphi(u)\right]du.
$$

This is `_call_price_from_integrals` in `pricing/european_gl.py` (the boxed line); `I1`/`I2` are the two sums. Puts come from put–call parity.

### 2.2 The characteristic function — and why the *Cui* form

The textbook Heston CF has a complex $\log\!\big(\tfrac{1-g e^{dT}}{1-g}\big)$ term with a **branch cut**: as $T$ grows the argument winds around the origin and naive `log` jumps by $2\pi i$ → discontinuous, wrong long-maturity prices.

Cui et al. (2016) rewrite it so the term becomes a numerically continuous $\log B \equiv D$. The CF (`models/heston_cf_cui.py`):

$$
\varphi(u)=\exp\Big\{ iu(\ln S_0 + (r-q)T) - \frac{T\kappa\bar v\rho\,iu}{\sigma} - v_0 A + \frac{2\kappa\bar v}{\sigma^2}D \Big\}
$$

intermediates:

$$
\xi = \kappa - \sigma\rho\,iu,\qquad
d = \sqrt{\xi^2 + \sigma^2(u^2+iu)},
$$
$$
A_1=(u^2+iu)\sinh\tfrac{dT}{2},\quad
A_2=d\cosh\tfrac{dT}{2}+\xi\sinh\tfrac{dT}{2},\quad
A=\frac{A_1}{A_2},
$$
$$
D=\ln d + \frac{(\kappa-d)T}{2} - \ln\!\Big(\frac{d+\xi}{2}+\frac{d-\xi}{2}e^{-dT}\Big).
$$

Chosen for two reasons: **stability** ($D$ never crosses the branch cut) and **differentiability** (clean enough to hand-differentiate w.r.t. all 5 params — the whole point for calibration).

> ⚠️ **Foreshadowing the bug:** $A=A_1/A_2$ is **not** in log form — it carries raw $\sinh/\cosh$ of $dT/2$. $D$ is log-stable; $A$ is not. (Section 6.)

### 2.3 Integral → number: Gauss–Legendre quadrature

$I_1,I_2$ run to $\infty$ but the integrand decays fast. Cui truncate at $\bar u=200$ and use **64-node Gauss–Legendre**:

$$\int_0^{200} g(u)\,du \approx \sum_{n=1}^{64} w_n\, g(u_n).$$

GL is *spectrally accurate* for smooth integrands — 64 nodes buys ~$10^{-8}$ price accuracy. Nodes/weights are precomputed once at import (`_U_NODES`, `_W_SCALED`). A price = evaluate $\varphi$ at 64 points, two weighted sums. Microseconds.

Let's price one contract.

In [ ]:
import sys, os
# Ensure repo root is importable (adjust if running from elsewhere)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
from pricing.european_gl import heston_call_gl, heston_call_price_and_gradient

S0, K, r, T, q = 100.0, 100.0, 0.045, 1.0, 0.0
v0, kappa, theta, sigma, rho = 0.04, 2.0, 0.04, 0.5, -0.7

price = heston_call_gl(S0, K, r, T, v0, kappa, theta, sigma, rho, q)
print(f"ATM 1y Heston call price: {price:.4f}")

### 2.4 The analytic gradient (Cui Theorem 1) → the Jacobian

The move that makes this method *fast*. Most calibrators get the Jacobian by **finite differences** (bump each param, re-price): 5 extra full pricings per contract per step. Cui give the gradient in closed form.

Because $\varphi=e^{(\text{exponent})}$, its parameter-derivative factorizes:

$$\frac{\partial\varphi}{\partial\theta_j} = \varphi\cdot h_j(u),$$

where $h_j$ is the derivative of the exponent — a known algebraic expression (`heston_cf_and_gradient`). E.g. $h_{v_0}=-A$; the rest follow from differentiating $A,D$ w.r.t. $\rho,\sigma$ (and reusing those for $\kappa,\bar v$). Push through the pricing integral:

$$
\frac{\partial C}{\partial\theta_j} = \frac{e^{-rT}}{\pi}\Big( I_{1,j} - K\,I_{2,j}\Big),\qquad
I_{1,j}=\int \mathrm{Re}\!\Big[\tfrac{K^{-iu}}{iu}\varphi(u-i)\,h_j(u-i)\Big]du.
$$

So `heston_call_price_and_gradient` returns price **and** $\nabla_\theta C$ in one sweep — ~2 pricings of cost, not 6. Over a chain this is the 10–16× speedup. Let's verify it against finite differences.

In [ ]:
price, grad = heston_call_price_and_gradient(S0, K, r, T, v0, kappa, theta, sigma, rho, q)
labels = ["v0", "kappa", "theta", "sigma", "rho"]

# Finite-difference check
base = np.array([v0, kappa, theta, sigma, rho], float)
fd = np.empty(5)
eps = 1e-6
for j in range(5):
    bump = base.copy(); bump[j] += eps
    fd[j] = (heston_call_gl(S0, K, r, T, *bump, q) - price) / eps

print(f"{'param':6} {'analytic':>14} {'finite-diff':>14} {'abs err':>12}")
for j in range(5):
    print(f"{labels[j]:6} {grad[j]:14.6f} {fd[j]:14.6f} {abs(grad[j]-fd[j]):12.2e}")

---
## 3. The optimizer: nonlinear least squares

### 3.1 Objective and residual vector

Define the **residual vector** (`heston_residuals`):

$$
r_i(\theta) = w_i\big(C_{\text{model}}(\theta;K_i,T_i) - C^{*}_i\big),\qquad
f(\theta)=\tfrac12\|r(\theta)\|^2 .
$$

Two design choices:

**(a) Price residuals, not IV residuals** — because the analytic gradient gives price sensitivities directly. But raw price errors over-weight expensive deep-ITM options. So…

**(b) Vega weighting** — $w_i = 1/\text{vega}_i$. Since $\Delta C \approx \text{vega}\cdot\Delta\sigma_{\text{IV}}$, dividing by vega turns a price residual into an **approximate IV residual**: numerical convenience of price-space, economic behavior of IV-space. Weights are constant in $\theta$, so the analytic Jacobian stays exact (each row just scaled by $w_i$).

(A soft Feller penalty residual is appended but `FELLER_WEIGHT = 0.0` — off, see memory `heston-calibration-rho-phase4`.)

### 3.2 Gauss–Newton: the core idea

Linearize the residual around the current guess $\theta_k$:

$$r(\theta_k+\delta)\approx r(\theta_k) + J_k\,\delta,\qquad J_{ij}=\frac{\partial r_i}{\partial\theta_j}.$$

$J$ is the stacked analytic gradients from 2.4 (rows = contracts, columns = 5 params). Minimizing $\tfrac12\|r_k+J_k\delta\|^2$ over $\delta$ gives the **normal equations**:

$$\boxed{\ (J_k^\top J_k)\,\delta = -J_k^\top r_k\ }\qquad\text{(Gauss–Newton step)}.$$

Pricing-view interpretation: $J_k^\top r_k$ is the misfit gradient ("which params reduce total mispricing fastest"); $J_k^\top J_k$ approximates curvature. Solve, step, repeat.

Gauss–Newton is fast near the solution but **unstable far away**: if $J^\top J$ is near-singular (e.g. $\kappa$ and $\bar v$ are notoriously correlated), $\delta$ explodes.

### 3.3 Levenberg–Marquardt: damping the step

LM adds damping $\lambda$:

$$\boxed{\ (J_k^\top J_k + \lambda\, \mathrm{diag}(J_k^\top J_k))\,\delta = -J_k^\top r_k\ }$$

- $\lambda\to 0$: pure Gauss–Newton (fast, trusts the quadratic model).
- $\lambda\to\infty$: $\delta \to -\tfrac1\lambda\nabla f$, a tiny **gradient-descent** step (safe, slow).

LM **adapts** $\lambda$: step reduced $f$ → shrink $\lambda$ (bolder); step worsened $f$ → grow $\lambda$ (cautious) and retry. It slides between aggressive Newton and safe gradient descent based on whether the local quadratic picture is trustworthy — which is why Heston calibration converges from a broad fixed starting box without a hand-tuned warm start.

### 3.4 Trust-Region-Reflective: what scipy actually runs

The code calls `scipy.optimize.least_squares(..., method='trf')`. TRF is the trust-region cousin of LM, picked for two reasons.

**Duality with LM.** Instead of "add $\lambda$ to the diagonal," a trust-region method minimizes the linear model inside a ball of radius $\Delta$:

$$\min_\delta\ \tfrac12\|r_k+J_k\delta\|^2 \quad\text{s.t.}\ \|\delta\|\le \Delta.$$

By Lagrangian duality this is *exactly* the LM step for some $\lambda$ — the damping $\lambda$ is the multiplier of the radius constraint. TRF manages $\Delta$ instead of $\lambda$, expanding/shrinking based on how well the model predicted the actual reduction.

**Why TRF specifically: bounds.** Heston params have hard limits ($v_0,\bar v>0$, $-1<\rho<1$, $\sigma$ capped). Plain LM is unconstrained; TRF enforces box bounds via **reflective transformations** that keep every iterate strictly feasible.

Stopping tolerances `ftol=gtol=xtol=1e-10` mirror the paper's $\varepsilon_1,\varepsilon_2,\varepsilon_3$: stop when cost stops improving, **or** the gradient $J^\top r\approx 0$ (first-order optimality), **or** the step $\delta$ is negligible.

The `svd(J_augmented)` in the earlier traceback is TRF solving the damped step via an SVD of $J$ (stabler than forming $J^\top J$) — so one NaN anywhere in $J$ kills the whole solve.

---
## 4. The calibration loop, start to finish

```
0. Build the calibration universe  (which quotes to fit)
1. theta <- initial guess (clamped into bounds)
2. repeat:
     a. for each contract i:  price C_i(theta) and gradient grad C_i(theta)   <- 2.3-2.4, ONE GL sweep
     b. assemble residuals r_i = w_i(C_i - C*_i)  and Jacobian J              <- 3.1
     c. solve damped/trust-region step  (J^T J + lambda D) delta = -J^T r     <- 3.3-3.4
     d. trial theta+delta; cost dropped -> accept & loosen; else reject & tighten
   until  ||step||, ||J^T r||, or dcost  <  1e-10
3. return theta*, loss = 0.5||r(theta*)||^2
```

Steps 2a–2d are scipy's `trf`; you supply 2a–2b via `residuals_fn` / `jacobian_fn` in `calibrate_heston.py`. Step 0 is the service layer's universe selection (next section), which matters a lot in practice.

---
## 5. How you actually use it

The whole thing is one service call (`services/calibration_service.py`):

```python
from services.calibration_service import calibrate_option_chain
result, calib_df = calibrate_option_chain(
    filtered_df,
    rate_curve=rate_curve,
    initial_guess=GUESS,   # HestonParameters(v0,kappa,theta,sigma,rho)
    bounds=BOUNDS,         # list of 5 (lo,hi) tuples
)
# result.params -> fitted HestonParameters
# result.loss   -> 0.5||r||^2 at the optimum
```

Inside, in order:

1. **`select_calibration_universe`** — "calibrate tight, price broad." Keeps the **OTM leg only** per strike (OTM put below the forward, OTM call above), near-ATM per expiry, a moneyness band, OI floor, and a **maturity window** (`min_maturity`/`max_maturity`). OTM-only keeps the clean skew/smile info and drops wide-spread near-intrinsic ITM noise. *(Also your lever for the long-dated SPX problem — Section 6.)*
2. **De-Americanization** — quotes converted to European-equivalent so the European CF pricer is consistent. For SPX (index, European) this is a pass-through.
3. **Per-row rates** — each contract's `r` interpolated from the SOFR/OIS curve by maturity.
4. **Run LM/TRF** — `calibrate_heston` with the analytic Jacobian (all-European universe).
5. **Cache** — persisted to `results/calibrations/`.

**Reading the output:** `result.loss` is the half-sum of squared vega-weighted price residuals — roughly a sum of squared IV errors; a clean equity chain lands per-contract RMS IV error in the low single-digit vol points. Feed `result.params` into pricing/analytics to build the model vol surface vs market.

**Main dials:** `bounds` (widen $\sigma$ for high-vol single names; keep $\rho<0$ for equity skew), `initial_guess` ($v_0\approx$ short-ATM variance, $\bar v\approx$ long-ATM variance), `max_maturity` / `contracts_per_expiry` / `atm_band`, and `weight_scheme` (`"vega"` default, `"none"`, `"inv_spread"`).

---
## 6. A failure that *was* here, and the fix (SPX long-dated LEAPS)

> **Status: FIXED** in `models/heston_cf_cui.py`. This section keeps the diagnosis (it's instructive) and then shows the log-safe reformulation that resolved it. The cell below now runs clean where it used to crash.

**The bug.** The price was built from $A=A_1/A_2$ with $A_1=(u^2+iu)\sinh\tfrac{dT}{2}$ and $A_2=d\cosh\tfrac{dT}{2}+\xi\sinh\tfrac{dT}{2}$.

- $\sinh\tfrac{dT}{2},\cosh\tfrac{dT}{2}$ grow like $e^{\mathrm{Re}(d)\,T/2}$. At the **high GL nodes** ($u$ up to 200) and **long maturities** ($T\approx 4.5$–$5.5$ yr for SPX LEAPS), $\mathrm{Re}(d)\,T/2$ approaches the double-precision ceiling (~709 in the exponent).
- The extra $u^2\approx 40{,}000$ factor in $A_1$ tipped it over → $A_1=\infty$ → $A=\infty$ → $\varphi=e^{-v_0 A+\dots}=$ **NaN**.
- That NaN landed in $r$ and $J$; TRF's `svd(J)` then raised *"array must not contain infs or NaNs."*

It was a weakness of the *implementation*, not of LM/TRF: Cui stabilized $D$ into log form but left $A=A_1/A_2$ in raw $\sinh/\cosh$ form — fine up to ~2-year maturities (NVDA never tripped it), but it overflowed on SPX's 4–5.5-year tail once the optimizer pushed $\sigma$ above ~1.3–1.6.

**The fix — normalise the $\cosh$ out (work in $\tanh$).** $A_1$ and $A_2$ only ever appear as the ratio $A=A_1/A_2$ (and inside $\partial A/A_2,\ \partial D/A_2$). Both carry the **same** $\cosh\tfrac{dT}{2}$ factor, so it cancels identically. Divide both by $\cosh\tfrac{dT}{2}$ up front:

$$
A=\frac{A_1}{A_2}=\frac{(u^2+iu)\sinh\tfrac{dT}{2}}{d\cosh\tfrac{dT}{2}+\xi\sinh\tfrac{dT}{2}}
=\frac{(u^2+iu)\,\tanh\tfrac{dT}{2}}{d+\xi\,\tanh\tfrac{dT}{2}}.
$$

$\tanh\tfrac{dT}{2}$ is **bounded** ($|\tanh|\le1$) because the principal square root gives $\mathrm{Re}(d)\ge0\Rightarrow\mathrm{Re}(\tfrac{dT}{2})\ge0$. The huge exponentials never materialise — nothing to overflow — and since it's the *same expression*, every price and gradient is unchanged to machine precision (verified: max $|\Delta\varphi|\sim10^{-15}$, max $|\Delta h|\sim10^{-10}$ vs the old code on 400 random draws; NVDA fits are bit-identical). The identical $\cosh$-cancellation is applied to the $\rho$- and $\sigma$-derivatives too.

In [ ]:
import numpy as np
np.seterr(all="ignore")
from pricing.european_gl import heston_call_price_and_gradient

# The exact param point / maturities that USED to overflow on SPX.
v0b, kappab, thetab, rhob = 0.08, 0.78, 0.095, -0.12
S, K, r_, q_ = 7500.0, 7500.0, 0.045, 0.0

print("After the tanh reformulation — price & gradient stay finite even far past the old ceiling:")
print(f"{'T (yrs)':>8} {'sigma':>6} {'sigma*T':>8} {'price':>14} {'grad finite':>12}")
for T in [1.0, 2.0, 3.0, 4.475, 5.47]:
    for sigma in [1.0, 2.0, 3.0]:
        p, g = heston_call_price_and_gradient(S, K, r_, T, v0b, kappab, thetab, sigma, rhob, q_)
        ok = np.isfinite(p) and np.all(np.isfinite(g))
        print(f"{T:8} {sigma:6} {sigma*T:8.1f} {p:14.4f} {str(ok):>12}")

print("\nEvery row finite (sigma*T up to ~16). Pre-fix these NaN'd and crashed the calibration.")

### The fix that was applied (and the alternatives)

**Applied:** *rewrite $A$ (and its $\rho,\sigma$ derivatives) in $\cosh$-normalised / $\tanh$ form* — the same log-safe spirit already used for $D$ — so the huge $\sinh/\cosh$ magnitudes never materialise. See `_intermediates` in `models/heston_cf_cui.py`. This is exact (no accuracy trade-off) and needs no change to the calibration universe.

**Alternatives not needed once the CF is stable** (still useful as belt-and-suspenders):
- **Cap `max_maturity`** in the calibration universe to drop the 4y+ tail.
- **Lower `_U_MAX`** or scale the GL nodes with $T$ so $u^2\,e^{\mathrm{Re}(d)T/2}$ stays bounded.

With the fix in place, the live SPX chain (5391 contracts, $T$ to 4.5y) calibrates end-to-end instead of raising `ValueError`.

---

**Pipeline in one line:** Heston SDE → closed-form CF (Cui: stable + differentiable, now overflow-safe via $\tanh$) → Fourier price via GL quadrature → analytic gradient = Jacobian → vega-weighted price residuals → Gauss–Newton step → LM damping → Trust-Region-Reflective with bounds → fitted $\theta$.

---
# 7. Root cause: why $\kappa$ pins to the upper bound (NVDA)

A recurring NVDA symptom: the fitted mean-reversion speed $\kappa$ lands exactly on the upper bound (`10.0`). On a live NVDA snapshot this run produced:

$$
v_0=0.139,\quad \kappa=\mathbf{10.0}\ \text{(upper bound)},\quad \bar v=0.184,\quad \sigma=\mathbf{0.05}\ \text{(lower bound)},\quad \rho=\mathbf{-0.10}\ \text{(upper bound)}.
$$

So it isn't only $\kappa$ — **three** parameters hit bounds at once. That is the tell: this is a *weak-identification / degenerate-corner* problem, not a coding bug. Below, four interlocking mechanisms, each with evidence from this chain.

## 7.1 One-sentence cause

> **$\kappa$ enters option prices only as a weak, second-order signal (the *curvature* of the variance term structure and the *maturity-decay* of the skew). The objective is therefore almost flat in $\kappa$; its gradient is small but keeps one sign, so Trust-Region-Reflective slides $\kappa$ along that flat valley until it parks against whichever bound the sign points to — here, the upper one.**

Everything else is the detailed "why."  

## 7.2 Mechanism A — $\kappa$ is a *curvature* parameter, and its signal saturates

To leading order, an ATM option's price is set by the **expected integrated variance** over its life. Under Heston the variance mean-reverts deterministically in expectation:

$$
\mathbb E[v_t] = \bar v + (v_0-\bar v)e^{-\kappa t}
\ \Longrightarrow\
\underbrace{\frac1T\mathbb E\!\Big[\int_0^T v_t\,dt\Big]}_{\approx\ \text{ATM implied variance}(T)}
= \bar v + (v_0-\bar v)\,\frac{1-e^{-\kappa T}}{\kappa T}.
$$

Read off the roles:

- $v_0$ sets the **short end**, $\bar v$ sets the **long end** — these are *level* parameters (big price impact).
- $\kappa$ only sets **how fast** the term structure travels from $v_0$ to $\bar v$ — it shapes the *curvature*, a far weaker lever.

And critically, the $\kappa$-factor $g(\kappa T)=\dfrac{1-e^{-\kappa T}}{\kappa T}$ **saturates**:

$$
g(\kappa T)\to 1\ (\kappa T\to 0),\qquad g(\kappa T)\to 0\ (\kappa T\to\infty),\qquad
\frac{\partial g}{\partial \kappa}=O\!\left(\frac1{\kappa}\right)\to 0.
$$

Once $\kappa T \gg 1$ for the bulk of the maturities, **changing $\kappa$ barely moves any price** — the gradient $\partial(\text{loss})/\partial\kappa$ decays toward zero but does not change sign, so the optimizer keeps inching $\kappa$ up forever and hits the bound. The cell below makes the saturation visible.

In [5]:
import numpy as np

v0, vbar = 0.139, 0.184      # NVDA-like: short var < long var (upward variance term structure)
T = np.array([0.05, 0.1, 0.25, 0.5, 1.0, 2.0])

def atm_var(kappa, T, v0, vbar):
    kT = kappa * T
    g = np.where(kT > 0, (1 - np.exp(-kT)) / np.where(kT==0, 1, kT), 1.0)
    return vbar + (v0 - vbar) * g

print("Model ATM implied *vol* sqrt((1/T)E[int v]) by maturity, for several kappa:")
print("T:        " + "  ".join(f"{t:6.2f}" for t in T))
prev = None
for kappa in [1, 2, 3, 5, 8, 10]:
    iv = np.sqrt(atm_var(kappa, T, v0, vbar))
    line = "  ".join(f"{x:6.3f}" for x in iv)
    delta = "" if prev is None else f"   max move vs prev kappa: {np.max(np.abs(iv-prev))*1e4:5.1f} bps"
    print(f"kappa={kappa:<3} {line}{delta}")
    prev = iv

print("\nNote how the term-structure curves for kappa=5,8,10 are nearly identical:")
print("the per-kappa change collapses to a few bps -> the loss is ~flat in kappa at the top end.")

Model ATM implied *vol* sqrt((1/T)E[int v]) by maturity, for several kappa:
T:          0.05    0.10    0.25    0.50    1.00    2.00
kappa=1    0.374   0.376   0.380   0.385   0.394   0.406
kappa=2    0.376   0.378   0.385   0.394   0.406   0.416   max move vs prev kappa: 112.4 bps
kappa=3    0.377   0.381   0.390   0.401   0.412   0.420   max move vs prev kappa:  64.6 bps
kappa=5    0.380   0.385   0.398   0.409   0.418   0.424   max move vs prev kappa:  83.7 bps
kappa=8    0.383   0.391   0.406   0.416   0.422   0.426   max move vs prev kappa:  77.5 bps
kappa=10   0.385   0.394   0.409   0.418   0.424   0.426   max move vs prev kappa:  36.0 bps

Note how the term-structure curves for kappa=5,8,10 are nearly identical:
the per-kappa change collapses to a few bps -> the loss is ~flat in kappa at the top end.


## 7.3 Mechanism B — the profile-loss valley runs *into* the bound

The clean test for "is $\kappa$ identified?" is a **profile loss**: fix $\kappa$, re-optimize the other four parameters, record the best achievable loss. If $\kappa$ were well-identified there'd be a clear interior minimum. Here is the actual NVDA result (274 calibration contracts):

| $\kappa$ | min loss | $v_0$ | $\bar v$ | $\sigma$ | $\rho$ |
|---:|---:|---:|---:|---:|---:|
| 0.5 | 189.59 | 0.160 | 0.234 | **0.050** | **−0.100** |
| 1   | 187.46 | 0.158 | 0.209 | **0.050** | **−0.100** |
| 2   | 184.09 | 0.154 | 0.197 | **0.050** | **−0.100** |
| 3   | 181.72 | 0.151 | 0.193 | **0.050** | **−0.100** |
| 5   | 178.92 | 0.147 | 0.188 | **0.050** | **−0.100** |
| 7   | 177.56 | 0.143 | 0.186 | **0.050** | **−0.100** |
| 9   | 176.89 | 0.140 | 0.184 | **0.050** | **−0.100** |
| 10  | **176.71** | 0.139 | 0.184 | **0.050** | **−0.100** |

Two things to see:

1. **Monotone, never interior.** The loss decreases *all the way* to $\kappa=10$ and would keep going — there is no interior minimum to stop at. From $\kappa=3$ to $\kappa=10$ the loss improves only **2.75%**; from $0.5$ to $10$, only **6.8%**. That is a flat valley with a gentle downslope into the bound. TRF faithfully follows it to the wall.
2. **$\sigma$ and $\rho$ are pinned at *their* bounds in every row** — the smile has collapsed (Mechanism D).

## 7.4 Mechanism C — vanishing $\kappa$-sensitivity $\Rightarrow$ near-singular $J^\top J$ $\Rightarrow$ boundary drift

Recall the optimizer step solves $(J^\top J + \lambda D)\,\delta = -J^\top r$, where $J$'s columns are $\partial C/\partial\theta_j$ over all contracts. The $\kappa$-column is tiny. Measured on this chain at a reasonable interior point $(v_0,\kappa,\bar v,\sigma,\rho)=(0.14,3,0.18,0.5,-0.4)$:

| param | $\lVert\partial C/\partial\theta_j\rVert$ | scaled by magnitude (elasticity) |
|---|---:|---:|
| $v_0$ | 379.1 | 53.1 |
| $\bar v$ | 329.2 | 59.3 |
| $\sigma$ | 13.3 | 6.6 |
| $\rho$ | 12.0 | 4.8 |
| **$\kappa$** | **2.85** | **8.6** |

$\kappa$'s raw price sensitivity is **~130×** smaller than $v_0$/$\bar v$. So the $\kappa\kappa$ entry of the Gauss-Newton curvature $J^\top J$ is minuscule and that direction is nearly a **null space** of the problem. Consequences:

- The step in the $\kappa$ direction is essentially undetermined by the data; it is set by the faint, persistently-signed residual gradient.
- TRF's reflective bound handling then **parks** the iterate on the active $\kappa$ bound and stops varying it (the projected gradient there is consistent with optimality).

This is the numerical face of the same identification problem: a flat objective direction shows up as a rank-deficient $J$.

## 7.5 Mechanism D — the smile collapse that makes it worse ($\sigma\to0$, $\rho\to$ bound)

On this NVDA snapshot the fit doesn't just lose $\kappa$ — it sends $\sigma\to0.05$ (lower bound) and $\rho\to-0.10$ (upper bound). That is the model **switching off its own smile**:

- The leading-order short-maturity ATM skew under Heston is $\propto \rho\sigma$. As $\sigma\to0$, the product $\rho\sigma\to0$ **regardless of $\rho$** — so $\rho$ becomes unidentified and floats to a bound arbitrarily.
- With $\sigma\approx0$ the variance process is nearly **deterministic** ($dv\approx\kappa(\bar v-v)\,dt$): no vol-of-vol, no curvature in the smile. Heston degenerates to a *time-dependent-but-deterministic* variance model.

Why does the optimizer choose this corner? The de-Americanized NVDA IV term structure on this snapshot is **jagged and non-monotonic** (e.g. ATM IV jumps $0.45\to0.38\to0.47\to0.36$ across adjacent maturities). That noise has no consistent smooth-smile explanation, so vega-weighted least squares does the *least-bad* thing: it abandons the smile ($\sigma\to0$) and fits only the average **level** term structure with $v_0,\bar v,\kappa$. But fitting a level term structure is exactly the regime where (Mechanism A) $\kappa$ wants to be as large as possible to snap from $v_0$ to $\bar v$ — so the smile collapse and the $\kappa$ blow-up reinforce each other.

## 7.6 The deeper reason — a cross-section barely contains $\kappa$

Step back: $\kappa$ is fundamentally a **time-series / dynamics** parameter — it governs *how variance evolves through time*. We are estimating it from a **single-day cross-section** of option prices. A snapshot pins down:

- the **level** of expected variance per maturity (term structure)  $\to v_0,\bar v$,
- the **shape** of the smile per maturity (skew, curvature)         $\to \rho,\sigma$,

but the **speed of mean reversion** only leaks in through (i) the *curvature* of the term structure and (ii) the *maturity-decay rate of the skew* — both second-order and easily traded off against $\bar v$ and $\sigma$. To actually pin $\kappa$ you need information a snapshot doesn't have: the realized-variance time series, or a maturity dimension that is simultaneously **long, dense, and clean** enough to resolve the curvature. NVDA (max $T\approx2.5$y, noisy de-Am IVs) is none of those. This is why $\kappa$ is the classic "fix it, don't fit it" Heston parameter.

## 7.7 What to do about it

Ordered roughly by leverage:

1. **Fix $\kappa$ (industry standard).** Hold $\kappa$ at an economically sensible value (typically $1$–$3$) and calibrate the other four. The repo already has the machinery — pass a tight `bounds` entry like `(1.5, 1.5)` for $\kappa$, or estimate it from realized variance via `calibration/historical_bounds.py` / `data_driven_bounds.py`.
2. **Tighten the $\kappa$ upper bound** to something plausible (e.g. $\le 5$). It won't fix identification, but it stops the corner solution from being absurd ($\kappa=10$ ⟺ a ~25-day variance half-life).
3. **Regularize $\kappa$ toward a prior** (Tikhonov / ridge): add a residual $\sqrt{\lambda_\kappa}\,(\kappa-\kappa_0)$ to the vector. A small $\lambda_\kappa$ supplies the missing curvature in $J^\top J$ and gives an interior minimum without distorting price fit much. (Note this is *different* from the Feller penalty, which is off for good reasons.)
4. **Fix the smile collapse first.** Clean the de-Americanized IV term structure (it's driving $\sigma\to0$): drop or down-weight the jagged long-dated NVDA quotes, enforce a liquidity/monotonicity screen, and reconsider vega weighting on the noisy wings. A coherent smile keeps $\sigma$ and $\rho$ off their bounds, which in turn stops feeding Mechanism A.
5. **Calibrate in a better-identified parameterization** (e.g. fit $\bar v$ and the half-life $\ln 2/\kappa$, or the product $\rho\sigma$ directly), so the optimizer steps along directions the data can actually resolve.

The single highest-value change is **#1** — because no amount of optimizer tuning can recover a parameter the cross-section does not contain.

### Summary

$\kappa$ hitting the bound is **weak identification surfacing as a boundary solution**, confirmed three ways on live NVDA: a profile loss that slides monotonically into the bound (only 2.75% gain over $\kappa\in[3,10]$), a $\kappa$-sensitivity ~130× below the level parameters (near-singular $J^\top J$), and a simultaneous $\sigma\to0,\ \rho\to$bound smile collapse that turns the fit into a pure level-term-structure problem — the exact regime that wants $\kappa\to\infty$. The fix is to stop fitting $\kappa$ from a cross-section and instead fix/regularize it, while cleaning the market IVs that trigger the smile collapse.